# SBA Loan Download and Preparation

This notebook downloads the SBA loan dataset and its supporting data dictionary into the workspace staging area under `data/raw/sba`.

It follows the repo's config-driven download pattern:

1. Create a destination folder under `data/raw`.
2. Write a YAML download config for the SBA CSV and Excel dictionary.
3. Call the project library to download the files.
4. Validate the downloaded artifacts.
5. Convert the Excel dictionary to a CSV in the same staging directory.
6. Leave the raw files ready for downstream EDA and dataset preparation.

The final output is a local, workspace-scoped SBA raw dataset and metadata dictionary, rather than downloading into the notebook's local working directory.


In [ ]:
from pathlib import Path

import pandas as pd
import yaml

from data_prep.download import download_from_config, verify_download_path

# Resolve the workspace root so downloads stay under the repo staging area.
workspace_root = Path.cwd().resolve()
if workspace_root.name == "notebooks":
    workspace_root = workspace_root.parent

raw_dir = (workspace_root / "data" / "raw").resolve()
raw_dir.mkdir(parents=True, exist_ok=True)

config_dir = (workspace_root / "configs").resolve()
config_dir.mkdir(parents=True, exist_ok=True)

dataset_name = "sba"
download_dir = (raw_dir / dataset_name).resolve()
download_dir.mkdir(parents=True, exist_ok=True)


def write_download_config(
    config_path: Path,
    url: str,
    *,
    dataset_name: str = "sba",
    dictionary_url: str | None = None,
) -> Path:
    """Create a YAML config for the repo's staged download location."""
    config = {
        "version": 1,
        "dataset_name": dataset_name,
        "operation": "download",
        "source": {
            "type": "http",
            "url": url,
        },
        "destination": {
            "path": str(download_dir),
            "format": "csv",
        },
    }
    if dictionary_url:
        config["data_dictionary"] = {
            "source_url": dictionary_url,
            "path": str(download_dir / "data_dictionary_raw.csv"),
            "filename": "data_dictionary_raw.csv",
            "format": "csv",
        }
    config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    return config_path


def update_download_config_with_files(config_path: Path, *, raw_file: Path, dictionary_file: Path, dictionary_url: str) -> None:
    """Persist the final downloaded file metadata back into the dataset config."""
    config = yaml.safe_load(config_path.read_text(encoding="utf-8")) or {}

    config["downloaded_files"] = [
        {
            "path": str(raw_file),
            "filename": raw_file.name,
            "size_bytes": raw_file.stat().st_size,
            "format": raw_file.suffix.lower().lstrip("."),
            "source_type": "raw_dataset",
        },
        {
            "path": str(dictionary_file),
            "filename": dictionary_file.name,
            "size_bytes": dictionary_file.stat().st_size,
            "format": dictionary_file.suffix.lower().lstrip("."),
            "source_type": "data_dictionary",
        },
    ]

    config["raw_data"] = {
        "path": str(raw_file),
        "filename": raw_file.name,
        "size_bytes": raw_file.stat().st_size,
        "format": raw_file.suffix.lower().lstrip("."),
    }

    config["data_dictionary"] = {
        "source_url": dictionary_url,
        "path": str(dictionary_file),
        "filename": dictionary_file.name,
        "size_bytes": dictionary_file.stat().st_size,
        "format": dictionary_file.suffix.lower().lstrip("."),
    }

    config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")


# Create the SBA raw-data config and download it through the library.
raw_data_url = "https://data.sba.gov/sites/default/files/uploaded_resources/FOIA_7a_FY2020_Present_asof_260630.csv"
raw_config_path = (config_dir / "sba_raw_data.yaml").resolve()
write_download_config(raw_config_path, raw_data_url, dataset_name=dataset_name)
raw_result = download_from_config(raw_config_path)
verify_download_path(raw_result)
print(f"Raw data download validated at: {raw_result}")

# Create the dictionary config and download it via the same library flow.
dictionary_url = "https://data.sba.gov/sites/default/files/uploaded_resources/7a_504_foia_data_dictionary.xlsx"
dictionary_config_path = (config_dir / "sba_data_dictionary.yaml").resolve()
write_download_config(dictionary_config_path, dictionary_url, dataset_name=dataset_name, dictionary_url=dictionary_url)
dictionary_result = download_from_config(dictionary_config_path)
verify_download_path(dictionary_result)

# The HTTP helper treats .xlsx files as zip archives and extracts them into a folder.
# Locate the actual workbook in the extracted directory before reading it.
possible_result = Path(dictionary_result)
excel_path = possible_result
if possible_result.is_dir():
    excel_candidates = sorted(possible_result.glob("*.xlsx")) + sorted(possible_result.rglob("*.xlsx"))
    if not excel_candidates:
        raise FileNotFoundError(f"No Excel workbook found in downloaded SBA dictionary directory: {possible_result}")
    excel_path = excel_candidates[0]

csv_dict_path = download_dir / "data_dictionary_raw.csv"
df_dict = pd.read_excel(excel_path, engine="openpyxl")
df_dict.to_csv(csv_dict_path, index=False)

raw_file = download_dir / "FOIA_7a_FY2020_Present_asof_260630.csv"
update_download_config_with_files(
    raw_config_path,
    raw_file=raw_file,
    dictionary_file=csv_dict_path,
    dictionary_url=dictionary_url,
)
update_download_config_with_files(
    dictionary_config_path,
    raw_file=raw_file,
    dictionary_file=csv_dict_path,
    dictionary_url=dictionary_url,
)

if possible_result.is_dir() and possible_result.exists():
    for path in sorted(possible_result.rglob("*"), reverse=True):
        if path.is_file() or path.is_symlink():
            path.unlink(missing_ok=True)
        elif path.is_dir():
            path.rmdir()

print(f"SBA files saved to: {download_dir}")
print(f"Dictionary rows: {len(df_dict)}")
print(f"Raw file: {(download_dir / 'FOIA_7a_FY2020_Present_asof_260630.csv').resolve()}")
print(f"Updated download config: {raw_config_path}")
print(f"Dictionary config metadata: {yaml.safe_load(raw_config_path.read_text(encoding='utf-8'))['data_dictionary']}")


Raw data download validated at: /home/rajiv/programming/kmds-dataset-util/data/raw/sba/FOIA_7a_FY2020_Present_asof_260630.csv
SBA files saved to: /home/rajiv/programming/kmds-dataset-util/data/raw/sba
Dictionary rows: 42
Raw file: /home/rajiv/programming/kmds-dataset-util/data/raw/sba/FOIA_7a_FY2020_Present_asof_260630.csv
